# Chapter 9: Hash Functions - Examples

Jupyter notebook demonstrating hash function properties, birthday attacks, Merkle-Damgård, sponge construction, and HMAC.

## 1. Preimage, second-preimage and collision

SHA-256/SHA-3 examples. A one-bit change in input changes the whole output (avalanche effect).

In [ ]:
import hashlib, binascii

def sha256(s):
    return hashlib.sha256(s.encode()).hexdigest()

m1 = "Hello World"
m2 = "hello world"
print(f"SHA-256('{m1}') = {sha256(m1)}")
print(f"SHA-256('{m2}') = {sha256(m2)}")
print(f"Avalanche: {sum(1 for a,b in zip(sha256(m1), sha256(m2)) if a!=b)} differing hex chars")

## 2. Birthday attack: expected number of hashes before a collision

For $n$ hash values, expected collision after about $\approx \sqrt{\pi n / 2}$ inputs. With 64-bit hashes that's about $2^{32}$ inputs.

In [ ]:
import math, random

def collision_experiment(bits, trials):
    """Return the average number of draws needed to find a collision."""
    total = 0
    for _ in range(trials):
        seen = set()
        count = 0
        while True:
            h = random.getrandbits(bits)
            count += 1
            if h in seen:
                break
            seen.add(h)
        total += count
    return total / trials

# Small bit-width: 12-bit hash
avg = collision_experiment(12, 2000)
theoretical = math.sqrt(math.pi * (2**12) / 2)
print(f"12-bit hash: empirical avg = {avg:.1f}, sqrt(pi*N/2) = {theoretical:.1f}")

## 3. Merkle-Damgård padding

Demonstrate the MD padding scheme: append `1`, then zeros, then a 64-bit length field.

In [ ]:
def md_pad(msg_bytes, block_bits=512):
    """Return bits of padded message per MD (like SHA-1/2)."""
    bits = ''.join(f'{b:08b}' for b in msg_bytes) + '1'
    len_bits = len(msg_bytes) * 8
    # pad until 64 bits before multiple of block
    while len(bits) % block_bits != block_bits - 64:
        bits += '0'
    bits += f'{len_bits:064b}'
    return bits

bits = md_pad(b"abc")
print(f"Padded 'abc' = {len(bits)} bits (multiple of 512), message length prefix = {len(bits)-448-64} bits")
print("Last 64 bits (length = 24):", bits[-64:])

## 4. Sponge construction (SHA-3/KECCAK)

The sponge absorbs input then squeezes output; rate $r$ + capacity $c$ define the security level.

In [ ]:
# SHAKE128: capacity c = 256 bits, security level c/2 = 128
print("SHAKE128: rate r=1344 bits (168 bytes), capacity c=256 bits, security = 128 bits")
print("SHA3-256: rate r=1088 bits (136 bytes), capacity c=512 bits, security = 256 bits")

# Generate arbitrary-length output with SHAKE
shake = hashlib.shake_128(b"message")
print("SHAKE128('message', 16 bytes):", shake.hexdigest(16))

## 5. HMAC construction

HMAC = Hash( (K \oplus opad) || Hash((K \oplus ipad) || m) ). Compare with plain keyed hash.

In [ ]:
def hmac_sha256(key, msg, block=64):
    if len(key) > block:
        key = hashlib.sha256(key).digest()
    key = key.ljust(block, b'\x00')
    ipad = bytes(x ^ 0x36 for x in key)
    opad = bytes(x ^ 0x5c for x in key)
    inner = hashlib.sha256(ipad + msg).digest()
    return hashlib.sha256(opad + inner).hexdigest()

key = b"my-secret-key"
msg = b"important message"
print("HMAC-SHA256:", hmac_sha256(key, msg))
print("Python hmac matches:", hmac_sha256(key, msg) == __import__('hmac').new(key, msg, hashlib.sha256).hexdigest())

## 6. Password hashing / brute force

Hash passwords and measure the difficulty of brute-forcing a keyspace.

In [ ]:
import hashlib, time

# Keyspace of an 8-char lowercase password:
keyspace = 26**8
print(f"8-char lowercase password keyspace = 26^8 = {keyspace:,.0f} ~ {math.log2(keyspace):.1f} bits")

# Time a single SHA-256, extrapolate to full keyspace
t0 = time.perf_counter()
for _ in range(10000): hashlib.sha256(b"guess").digest()
dt = (time.perf_counter() - t0) / 10000
print(f"~{dt*1e6:.1f} us per hash")
print(f"Time to try all (single core): {keyspace*dt/3600/24/365:.1e} years")